In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.46.3 -q


# Uncertainty-Weighted Drift (UWD) for Hallucination Detection

## Hypothesis
Semantic drift alone captures *what* changed between context and output. But it doesn't capture *how confident* the model was when generating. Combining drift with model uncertainty should improve detection.

## Key Idea
- **High drift + High confidence:** Model deliberately diverged → likely hallucination
- **High drift + Low confidence:** Model uncertain and drifted → possible hallucination
- **Low drift + High confidence:** Model stayed faithful → likely correct
- **Low drift + Low confidence:** Uncertain but stayed close → probably OK

## Metrics to Compute
1. **Mean Token Probability:** Average probability of generated tokens
2. **Min Token Probability:** Minimum probability token (weakest link)
3. **Perplexity:** Overall sequence perplexity

## Why This Could Work
- Drift measures semantic distance (what)
- Uncertainty measures model confidence (how sure)
- Together: "Did the model confidently produce something far from context?"

## Platform: Kaggle GPU (T4)
## Estimated Runtime: 1-2 hours

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Setup

In [ ]:
!pip install -q sentence-transformers transformers torch numpy scipy scikit-learn accelerate

In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
import pickle
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Load Dataset

In [ ]:
# Upload hotpot_rag_length_controlled_1000.json to Kaggle first!
with open(f'{WORK_DIR}/gemma/squad/hotpot_qwen25_rag_length_controlled_1000_binmatched.json') as f:
    dataset = json.load(f)

labels = np.array([s['hallucinated'] for s in dataset])

print(f"Loaded {len(dataset)} samples")
print(f"  Non-hallucinated: {(labels == 0).sum()}")
print(f"  Hallucinated: {(labels == 1).sum()}")

## Step 3: Compute SDS Baseline

In [ ]:
# Load sentence transformer for SDS
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Computing SDS baseline...")

E_context = []
E_output = []

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])
    E_context.append(embedder.encode(context_str, convert_to_numpy=True))
    E_output.append(embedder.encode(sample['answer'], convert_to_numpy=True))

E_context = np.array(E_context)
E_output = np.array(E_output)

SDS = np.linalg.norm(E_output - E_context, axis=1)
sds_auroc = roc_auc_score(labels, SDS)

print(f"\nSDS Baseline AUROC: {sds_auroc:.4f}")

## Step 4: Load Phi-3 Model

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # Paste your token here


In [ ]:
%pip install -U transformers accelerate

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()

model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # Changed from float16
    low_cpu_mem_usage=True
)

print("✓ Model loaded successfully!")

print(f"✓ Loaded {model_name}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


print(f"\nModel loaded!")



## Step 5: Define Uncertainty Extraction Function

We compute the probability of each token in the answer given the prompt + context. This tells us how "surprised" the model is by its own output.

In [ ]:
def compute_uncertainty_metrics(question, context, answer, model, tokenizer):
    """
    Compute uncertainty metrics for a generated answer.

    We feed [prompt + answer] and compute the probability of each answer token
    given the preceding context.

    Returns:
        - mean_prob: Average probability of generated tokens
        - min_prob: Minimum token probability (weakest link)
        - perplexity: Perplexity of the answer given the prompt
    """

    # Format prompt
    context_str = "\n".join(context)
    prompt = f"""Answer the question using the context below.

Context:
{context_str}

Question: {question}

Answer:"""

    # Full text = prompt + answer
    full_text = prompt + " " + answer

    # Tokenize
    prompt_tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    full_tokens = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=3000).to("cuda")

    prompt_len = prompt_tokens['input_ids'].shape[1]
    total_len = full_tokens['input_ids'].shape[1]
    answer_len = total_len - prompt_len

    if answer_len <= 1:
        # Answer too short
        return {
            'mean_prob': 1.0,
            'min_prob': 1.0,
            'perplexity': 1.0
        }

    # Forward pass
    with torch.no_grad():
        outputs = model(**full_tokens)
        logits = outputs.logits  # (1, seq_len, vocab_size)

    # Get logits for answer tokens (shifted by 1 for next-token prediction)
    answer_logits = logits[0, prompt_len-1:total_len-1, :]  # (answer_len, vocab_size)
    answer_token_ids = full_tokens['input_ids'][0, prompt_len:total_len]  # (answer_len,)

    # Compute probabilities
    probs = F.softmax(answer_logits, dim=-1)  # (answer_len, vocab_size)

    # Get probability of actual tokens
    token_probs = probs[torch.arange(answer_len), answer_token_ids]  # (answer_len,)

    # Compute perplexity
    # PPL = exp(-1/N * sum(log(p_i)))
    log_token_probs = torch.log(token_probs + 1e-10)
    avg_log_prob = log_token_probs.mean()
    perplexity = torch.exp(-avg_log_prob).item()

    return {
        'mean_prob': token_probs.mean().item(),
        'min_prob': token_probs.min().item(),
        'perplexity': min(perplexity, 1000.0)  # Cap for numerical stability
    }


# Test on one sample
print("Testing uncertainty extraction on one sample...")
test_metrics = compute_uncertainty_metrics(
    dataset[0]['question'],
    dataset[0]['context'],
    dataset[0]['answer'],
    model,
    tokenizer
)

print("\nExtracted metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

## Step 6: Extract Uncertainty Metrics for All Samples

In [ ]:
print("Extracting uncertainty metrics for all samples...")
print(f"Total samples: {len(dataset)}")
print("\nProgress saved every 100 samples.\n")

# Initialize storage
all_uncertainty = []

# Check for checkpoint
import os
checkpoint_file = 'uncertainty_checkpoint.pkl'
start_idx = 0

if os.path.exists(checkpoint_file):
    print("Found checkpoint, loading...")
    with open(checkpoint_file, 'rb') as f:
        checkpoint = pickle.load(f)
    all_uncertainty = checkpoint['all_uncertainty']
    start_idx = checkpoint['next_idx']
    print(f"Resuming from sample {start_idx}")

# Extract
for i in tqdm(range(start_idx, len(dataset)), initial=start_idx, total=len(dataset)):
    sample = dataset[i]

    try:
        metrics = compute_uncertainty_metrics(
            sample['question'],
            sample['context'],
            sample['answer'],
            model,
            tokenizer
        )
        all_uncertainty.append(metrics)

    except Exception as e:
        print(f"\nError on sample {i}: {e}")
        fallback = {
            'mean_prob': 1.0,
            'min_prob': 1.0,
            'perplexity': 1.0
        }
        all_uncertainty.append(fallback)

    # Save checkpoint every 100 samples
    if (i + 1) % 100 == 0:
        checkpoint = {
            'all_uncertainty': all_uncertainty,
            'next_idx': i + 1
        }
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint, f)
        print(f"\n  Checkpoint saved at sample {i + 1}")

print(f"\nExtraction complete! {len(all_uncertainty)} samples processed.")

## Step 7: Convert to Feature Arrays

In [ ]:
# Convert to numpy arrays
mean_prob = np.array([m['mean_prob'] for m in all_uncertainty])
min_prob = np.array([m['min_prob'] for m in all_uncertainty])
perplexity = np.array([m['perplexity'] for m in all_uncertainty])

print("Feature arrays created:")
print(f"  mean_prob: {mean_prob.shape}, range [{mean_prob.min():.3f}, {mean_prob.max():.3f}]")
print(f"  min_prob: {min_prob.shape}, range [{min_prob.min():.3f}, {min_prob.max():.3f}]")
print(f"  perplexity: {perplexity.shape}, range [{perplexity.min():.3f}, {perplexity.max():.3f}]")

## Step 8: Evaluate Uncertainty Features Individually

In [ ]:
print("="*70)
print("UNCERTAINTY FEATURE EVALUATION")
print("="*70)

print(f"\nSDS Baseline AUROC: {sds_auroc:.4f}")
print("-" * 50)

results = []

# Test each uncertainty feature
features = {
    'Mean Prob (neg)': -mean_prob,  # Lower prob = more uncertain = more hallucination
    'Min Prob (neg)': -min_prob,
    'Perplexity': perplexity
}

print("\nIndividual Uncertainty Features:")
for name, feat in features.items():
    # Handle NaN/Inf
    feat_clean = np.nan_to_num(feat, nan=0.0, posinf=1000.0, neginf=-1000.0)

    auroc = roc_auc_score(labels, feat_clean)
    status = "✅ BEATS SDS!" if auroc > sds_auroc else ""
    print(f"  {name}: {auroc:.4f} {status}")

    results.append({'Method': name, 'AUROC': auroc, 'Type': 'Uncertainty'})

# Add SDS
results.append({'Method': 'SDS Baseline', 'AUROC': sds_auroc, 'Type': 'Baseline'})

## Step 10: Combined Features with Logistic Regression

In [ ]:
print("\n" + "="*70)
print("COMBINED FEATURES (LOGISTIC REGRESSION)")
print("="*70)

scaler = StandardScaler()
lr = LogisticRegression(max_iter=1000, C=0.1, random_state=42)

# Clean all features
def clean_feature(f):
    return np.nan_to_num(f, nan=0.0, posinf=1000.0, neginf=-1000.0)

perplexity_clean = clean_feature(np.clip(perplexity, 1, 100))
mean_prob_clean = clean_feature(mean_prob)
min_prob_clean = clean_feature(min_prob)

# SDS + single uncertainty feature
print("\n--- SDS + Single Uncertainty Feature ---")
single_features = {
    'Perplexity': perplexity_clean,
    'Mean Prob': mean_prob_clean,
    'Min Prob': min_prob_clean
}

for name, feat in single_features.items():
    X = np.column_stack([SDS, feat])
    X_norm = scaler.fit_transform(X)

    scores = cross_val_score(lr, X_norm, labels, cv=5, scoring='roc_auc')
    mean_auroc = scores.mean()

    status = "✅" if mean_auroc > sds_auroc else ""
    print(f"SDS + {name}: {mean_auroc:.4f} ± {scores.std():.4f} {status}")

    results.append({'Method': f'SDS + {name}', 'AUROC': mean_auroc, 'Type': 'Combined'})

# SDS + all uncertainty features
print("\n--- SDS + All Uncertainty Features ---")
X_all = np.column_stack([
    SDS,
    perplexity_clean,
    mean_prob_clean,
    min_prob_clean
])
X_all_norm = scaler.fit_transform(X_all)

scores_all = cross_val_score(lr, X_all_norm, labels, cv=5, scoring='roc_auc')
mean_auroc_all = scores_all.mean()

status = "✅" if mean_auroc_all > sds_auroc else ""
print(f"SDS + All Uncertainty: {mean_auroc_all:.4f} ± {scores_all.std():.4f} {status}")
results.append({'Method': 'SDS + All Uncertainty', 'AUROC': mean_auroc_all, 'Type': 'Combined'})

# Only uncertainty features (no SDS)
print("\n--- Uncertainty Features Only (no SDS) ---")
X_unc = np.column_stack([
    perplexity_clean,
    mean_prob_clean,
    min_prob_clean
])
X_unc_norm = scaler.fit_transform(X_unc)

scores_unc = cross_val_score(lr, X_unc_norm, labels, cv=5, scoring='roc_auc')
mean_auroc_unc = scores_unc.mean()

status = "✅" if mean_auroc_unc > sds_auroc else ""
print(f"All Uncertainty (no SDS): {mean_auroc_unc:.4f} ± {scores_unc.std():.4f} {status}")
results.append({'Method': 'All Uncertainty (no SDS)', 'AUROC': mean_auroc_unc, 'Type': 'Uncertainty'})

## Step 11: Analysis - Uncertainty vs Hallucination

In [ ]:
print("\n" + "="*70)
print("UNCERTAINTY STATISTICS BY CLASS")
print("="*70)

print("\n{:<20} {:>15} {:>15} {:>10}".format('Metric', 'Non-Hall', 'Hallucinated', 'Diff'))
print("-" * 65)

metrics_to_analyze = {
    'Perplexity': perplexity_clean,
    'Mean Prob': mean_prob_clean,
    'Min Prob': min_prob_clean,
    'SDS': SDS
}

for name, metric in metrics_to_analyze.items():
    non_hall = metric[labels == 0]
    hall = metric[labels == 1]
    diff = hall.mean() - non_hall.mean()

    print(f"{name:<20} {non_hall.mean():>7.3f}±{non_hall.std():<6.3f} {hall.mean():>7.3f}±{hall.std():<6.3f} {diff:>+10.3f}")

## Step 12: Final Results Summary

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Sort by AUROC
results_sorted = sorted(results, key=lambda x: x['AUROC'], reverse=True)

print(f"\n{'Method':<35} {'AUROC':>10} {'vs SDS':>10} {'Type':>15}")
print("-" * 75)

for r in results_sorted:
    diff = r['AUROC'] - sds_auroc
    diff_str = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    status = "✅" if diff > 0.005 else ""
    print(f"{r['Method']:<35} {r['AUROC']:>10.4f} {diff_str:>10} {r['Type']:>15} {status}")

# Best method
best = results_sorted[0]
print(f"\n{'='*75}")
print(f"🏆 BEST METHOD: {best['Method']}")
print(f"   AUROC: {best['AUROC']:.4f}")
print(f"   Improvement over SDS: {best['AUROC'] - sds_auroc:+.4f}")

if best['AUROC'] > sds_auroc + 0.01:
    print(f"\n✅ SIGNIFICANT IMPROVEMENT! Uncertainty helps!")
elif best['AUROC'] > sds_auroc:
    print(f"\n⚠️ Marginal improvement. May need refinement.")
else:
    print(f"\n❌ No improvement over SDS baseline.")

## Step 13: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. AUROC comparison bar chart
ax = axes[0, 0]
top_methods = results_sorted[:7]
methods = [r['Method'][:25] for r in top_methods]
aurocs = [r['AUROC'] for r in top_methods]
colors = ['green' if a > sds_auroc else 'red' if r['Method'] == 'SDS Baseline' else 'gray'
          for a, r in zip(aurocs, top_methods)]

bars = ax.barh(methods[::-1], aurocs[::-1], color=colors[::-1], alpha=0.7, edgecolor='black')
ax.axvline(sds_auroc, color='red', linestyle='--', linewidth=2, label=f'SDS ({sds_auroc:.3f})')
ax.set_xlabel('AUROC', fontsize=12)
ax.set_title('Top Methods by AUROC', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='x')

# 2. Perplexity distribution
ax = axes[0, 1]
ax.hist(perplexity_clean[labels==0], bins=30, alpha=0.7, label='Non-hallucinated', color='green', density=True)
ax.hist(perplexity_clean[labels==1], bins=30, alpha=0.7, label='Hallucinated', color='red', density=True)
ax.axvline(perplexity_clean[labels==0].mean(), color='green', linestyle='--', linewidth=2)
ax.axvline(perplexity_clean[labels==1].mean(), color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Perplexity', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Perplexity Distribution by Class', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 3. SDS vs Perplexity scatter
ax = axes[1, 0]
scatter = ax.scatter(SDS, perplexity_clean, c=labels, cmap='RdYlGn_r', alpha=0.5, s=20)
ax.set_xlabel('SDS (Surface Drift Score)', fontsize=12)
ax.set_ylabel('Perplexity', fontsize=12)
ax.set_title('SDS vs Perplexity', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Hallucinated')
ax.grid(alpha=0.3)

# 4. Mean Prob distribution
ax = axes[1, 1]
ax.hist(mean_prob_clean[labels==0], bins=30, alpha=0.7, label='Non-hallucinated', color='green', density=True)
ax.hist(mean_prob_clean[labels==1], bins=30, alpha=0.7, label='Hallucinated', color='red', density=True)
ax.axvline(mean_prob_clean[labels==0].mean(), color='green', linestyle='--', linewidth=2)
ax.axvline(mean_prob_clean[labels==1].mean(), color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Mean Token Probability', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Mean Token Probability Distribution by Class', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('uncertainty_drift_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved uncertainty_drift_results.png")

## Step 14: Save Results

In [ ]:
# Save all results
results_to_save = {
    'all_uncertainty': all_uncertainty,
    'mean_prob': mean_prob,
    'min_prob': min_prob,
    'perplexity': perplexity,
    'SDS': SDS,
    'labels': labels,
    'sds_auroc': sds_auroc,
    'results': results
}

with open('hotpot_phi3_uncertainty_results.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print("✓ Saved hotpot_phi3_uncertainty_results.pkl")

# Save CSV summary
import pandas as pd
df = pd.DataFrame(results_sorted)
df['beats_sds'] = df['AUROC'] > sds_auroc
df['improvement'] = df['AUROC'] - sds_auroc
df.to_csv('hotpot_phi3_uncertainty_results.csv', index=False)

print("✓ Saved hotpot_phi3_uncertainty_results.csv")

# Clean up checkpoint
import os
if os.path.exists('uncertainty_checkpoint.pkl'):
    os.remove('uncertainty_checkpoint.pkl')
    print("✓ Cleaned up checkpoint")

print("\n" + "="*70)
print("UNCERTAINTY DRIFT EXPERIMENT COMPLETE!")
print("="*70)
print("\nFiles saved:")
print("  - hotpot_phi3_uncertainty_results.pkl")
print("  - hotpot_phi3_uncertainty_results.csv")
print("  - uncertainty_drift_results.png")

## Key Findings

**1. Uncertainty metrics outperform semantic drift alone**
- Perplexity achieves **0.8105 AUROC** vs SDS baseline of **0.6996**
- Combining SDS + Perplexity reaches **~0.8413 AUROC** (+14% improvement)

**2. Hallucinated answers show measurably higher uncertainty**
- **35% higher perplexity** (1.885 vs 1.390)
- **9.2% lower mean token probability** (0.755 vs 0.847)
- **65% lower minimum token probability** (0.028 vs 0.079)

**3. Interpretation**
Models generate hallucinations with **lower confidence** than faithful outputs. This refutes the "confident mistake" hypothesis — hallucinations are detectable through uncertainty signals.

**4. Practical implications**
- Uncertainty features work **independently** of semantic drift (0.8095 AUROC without SDS)
- Combining both signals provides **complementary information**
- Perplexity alone is a strong hallucination indicator

In [ ]:
import json
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Load the Ministral dataset
with open(f'{WORK_DIR}/phi/hotpot/hotpot_qwen25_rag_length_controlled_1000_binmatched.json', 'r') as f:
    data = json.load(f)

# Split by hallucination label (use 'hallucinated' key)
non_hall = [len(item['answer'].split()) for item in data if item['hallucinated'] == 0]
hall = [len(item['answer'].split()) for item in data if item['hallucinated'] == 1]

# Calculate statistics
print(f"Non-hallucinated: {np.mean(non_hall):.2f} ± {np.std(non_hall):.2f} (n={len(non_hall)})")
print(f"Hallucinated: {np.mean(hall):.2f} ± {np.std(hall):.2f} (n={len(hall)})")
print(f"Difference: {np.mean(hall) - np.mean(non_hall):.2f} words")

# T-test
t_stat, p_value = stats.ttest_ind(non_hall, hall)
print(f"T-test: p={p_value:.3f}")

# Length-only logistic regression
lengths = np.array([len(item['answer'].split()) for item in data]).reshape(-1, 1)
labels = np.array([item['hallucinated'] for item in data])

clf = LogisticRegression()
clf.fit(lengths, labels)
preds = clf.predict_proba(lengths)[:, 1]
auroc = roc_auc_score(labels, preds)
print(f"Length-only AUROC: {auroc:.3f}")
